In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

RESULTS_BASE = '../results/gso'
FOLDER_SUFFIX = '_many_scanners'
NUM_SCANNERS_LIST = [10, 20, 50, 100, 200, 500, 1000]
SEEDS = range(50, 200)

EXPERIMENT_DIRS = {
    'no_decouple': f'{RESULTS_BASE}/scout_no_decouple_many_scanners',
    'proper': f'{RESULTS_BASE}/scout_proper_many_scanners',
}

BASELINE_PATH = f'{RESULTS_BASE}/mean_script/results_gso'
SCOUT_PROPER_PATH = f'{RESULTS_BASE}/scout_proper/results_gso'


# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------

def _read_single_point_exp(results_path, seeds):
    """Read single_point_exp values across seeds from a results directory.

    Args:
        results_path: Directory containing results_{seed}.txt files.
        seeds: Iterable of seed values to read.

    Returns:
        List of single_point_exp float values.
    """
    values = []
    for seed in seeds:
        filepath = os.path.join(results_path, f'results_{seed}.txt')
        with open(filepath, 'r') as f:
            for line in f:
                key, val = line.strip().split(': ')
                if key == 'single_point_exp':
                    values.append(float(val))
    return values


def _collect_many_scanners_results(experiment_dir, folder_suffix, num_scanners_list, seeds):
    """Collect mean and std error of single_point_exp across scanner counts.

    Args:
        experiment_dir: Base directory for the experiment.
        folder_suffix: Suffix for results folders.
        num_scanners_list: List of scanner counts.
        seeds: Iterable of seed values.

    Returns:
        tuple: (means, std_errors) as numpy arrays.
    """
    means = []
    std_errors = []
    for num_scanners in num_scanners_list:
        results_path = f'{experiment_dir}/results{folder_suffix}_{num_scanners}'
        values = _read_single_point_exp(results_path, seeds)
        means.append(np.mean(values))
        std_errors.append(np.std(values) / np.sqrt(len(values)))
    return np.array(means), np.array(std_errors)


In [ ]:
# ---------------------------------------------------------------------------
# Collect results
# ---------------------------------------------------------------------------

# Many-scanners results for both methods
no_decouple_means, no_decouple_errs = _collect_many_scanners_results(
    EXPERIMENT_DIRS['no_decouple'], FOLDER_SUFFIX, NUM_SCANNERS_LIST, SEEDS
)
proper_means, proper_errs = _collect_many_scanners_results(
    EXPERIMENT_DIRS['proper'], FOLDER_SUFFIX, NUM_SCANNERS_LIST, SEEDS
)

# Baseline (input-agnostic, no std)
baseline_values = _read_single_point_exp(BASELINE_PATH, SEEDS)
baseline_mean = np.mean(baseline_values)

# SCOUT proper without many-scanners (reference horizontal line)
scout_proper_values = _read_single_point_exp(SCOUT_PROPER_PATH, SEEDS)
scout_proper_mean = np.mean(scout_proper_values)
scout_proper_err = np.std(scout_proper_values) / np.sqrt(len(scout_proper_values))

# ---------------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------------

x = np.array(NUM_SCANNERS_LIST)

fig, ax = plt.subplots(figsize=(10 / 1.2, 6 / 1.2))

ax.errorbar(
    x, no_decouple_means / baseline_mean, yerr=no_decouple_errs / baseline_mean,
    linewidth=2.5, markersize=8, capsize=5, capthick=2,
    label='Ours without Decoupling', color='#FF6B6B',
    markeredgecolor='black', markeredgewidth=0.5,
)

ax.errorbar(
    x, proper_means / baseline_mean, yerr=proper_errs / baseline_mean,
    linewidth=2.5, markersize=8, capsize=5, capthick=2,
    label='Ours with Decoupling', color='#4ECDC4',
    markeredgecolor='black', markeredgewidth=0.5,
)

# SCOUT proper (no many-scanners) as reference horizontal line
scout_proper_norm = np.full_like(x, scout_proper_mean / baseline_mean, dtype=float)
scout_proper_err_norm = np.full_like(x, scout_proper_err / baseline_mean, dtype=float)
ax.errorbar(
    x, scout_proper_norm, yerr=scout_proper_err_norm,
    linewidth=2.5, markersize=8, capsize=5, capthick=2,
    label='Ours with Decoupling (no view-invariant)', color='#45B7D1',
    markeredgecolor='black', markeredgewidth=0.5, linestyle='--',
)

ax.set_xlabel('Number of View-Invariant Models During Training',
              fontsize=14, fontweight='bold', labelpad=10)
ax.set_ylabel('Regret as a Percent of \n Input Agnostic Baseline \n (Lower is Better)',
              fontsize=14, fontweight='bold', labelpad=10)
ax.set_title('Performance over Viewpoint-Dependent Models \n when Trained with All Reconstruction Models',
             fontsize=16, fontweight='bold', pad=20)
ax.set_xscale('log')
ax.legend(fontsize=12, loc='best', framealpha=0.9, edgecolor='black')
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

# ---------------------------------------------------------------------------
# Print summary
# ---------------------------------------------------------------------------

print(f"\nSCOUT proper (no view-invariant): {scout_proper_mean:.6f} +/- {scout_proper_err:.6f}")
print(f"Baseline (input-agnostic) mean:   {baseline_mean:.6f}")
print(f"\nNo-decouple means: {no_decouple_means}")
print(f"Proper means:      {proper_means}")
